In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("Taxa-bronze").getOrCreate()

metadata_df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "false") \
    .option("sep", "\t") \
    .load("s3a://inaturalist-open-data/taxa.csv.gz")

ingested_df = metadata_df.withColumn(
    "ingestion_timestamp",
    F.current_timestamp()
).withColumn(
    "source_file_path",
    F.col("_metadata.file_path")
).withColumn(
    "snapshot_date",
    F.current_date()
)

ingested_df.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable("philippinebiodiveristyplatform.bronze.taxa")